# AIMO 3 - Harmony TIR (Tool-Integrated Reasoning)

Based on the way-to-30-50 approach:
- **Harmony Protocol**: Native GPT-OSS message format
- **Jupyter Kernel**: Stateful Python execution
- **Token-level streaming**: Direct token ID processing
- **Early stopping**: Majority voting with consistency threshold

Reference: way-to-30-50-gpt-oss-w-python-tool.ipynb

In [ ]:
import time
import numpy as np
import os

start_time = time.time()
final_cutoff_time = start_time + (4 * 60 + 55) * 60  # 4h 55m

In [ ]:
import subprocess

uninstall_proc = subprocess.Popen(
    ["pip", "uninstall", "--yes", "tensorflow", "matplotlib", "keras", "scikit-learn"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

In [ ]:
%%time
!find /kaggle/usr/lib -type f -print0 | xargs -0 -P 32 -n 500 cat > /dev/null

In [ ]:
def cache_model(path, exts=(".bin", ".pt", ".safetensors"), num_workers=None, chunk_mb=256):
    """Pre-read model weight files into OS page cache."""
    import os
    import multiprocessing
    import time
    from concurrent.futures import ThreadPoolExecutor, as_completed

    def warmup_file(fpath):
        chunk_size = chunk_mb * 1024 * 1024
        total = 0
        with open(fpath, "rb") as f:
            while True:
                data = f.read(chunk_size)
                if not data:
                    break
                total += len(data)
        return fpath, total

    if os.path.isdir(path):
        files = [
            os.path.join(root, name)
            for root, _, names in os.walk(path)
            for name in names
            if name.endswith(exts)
        ]
        files.sort()
    else:
        files = [path]

    if not files:
        raise ValueError(f"No model files found under: {path}")

    if num_workers is None:
        try:
            num_workers = min(multiprocessing.cpu_count(), 8)
        except Exception:
            num_workers = 4

    print(f"[cache_model] {len(files)} file(s), {num_workers} worker(s)")
    t0 = time.time()
    total_bytes = 0

    with ThreadPoolExecutor(max_workers=num_workers) as pool:
        futures = {pool.submit(warmup_file, f): f for f in files}
        for i, fut in enumerate(as_completed(futures), 1):
            fpath, n = fut.result()
            total_bytes += n
            print(f"[{i}/{len(files)}] cached {os.path.basename(fpath)}")

    elapsed = time.time() - t0
    gb = total_bytes / 1024**3
    print(f"[cache_model] total read ≈ {gb:.2f} GB in {elapsed:.2f}s")
    return total_bytes


cache_model("/kaggle/input/gpt-oss-120b/transformers/default/1", num_workers=16, chunk_mb=1024)

In [ ]:
%%time
# Copy vLLM compile cache if available
import os
if os.path.exists("/kaggle/input/gpt-oss-120b-cache-compile/torch_compile_cache"):
    !mkdir -p /root/.cache/vllm/
    !cp -r /kaggle/input/gpt-oss-120b-cache-compile/torch_compile_cache /root/.cache/vllm/

In [ ]:
uninstall_proc.wait()

In [ ]:
subprocess.run(["ls", "/kaggle/usr/lib/pip_install_aimo3_1/tiktoken_encodings"])

In [ ]:
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TIKTOKEN_ENCODINGS_BASE"] = "/kaggle/usr/lib/pip_install_aimo3_1/tiktoken_encodings"

# Python Tool with Jupyter Kernel

In [ ]:
%%writefile local_python_tool.py
"""Python tool using Jupyter kernel for stateful execution."""
import ast
import math
import os
import queue
import random
import re
import threading
from abc import ABC, abstractmethod
from collections import Counter, defaultdict
from itertools import combinations, permutations, product
from typing import Any, AsyncIterator
from uuid import UUID, uuid4

import numpy as np
import sympy as sp
from sympy import *  # noqa: F403,F401
from openai_harmony import (
    Author,
    Content,
    Message,
    Role,
    TextContent,
    ToolNamespaceConfig,
 )


_TOOL_PRELUDE = """
import math
import random
from itertools import combinations, permutations, product
from collections import defaultdict, Counter
import numpy as np
import sympy as sp
from sympy import *
""".strip() + "\n"


class AIMOToolkit:
    """Lightweight math helpers for AIMO3 tool runs."""

    @staticmethod
    def modular_power(a: int, b: int, m: int) -> int:
        return pow(a, b, m)

    @staticmethod
    def smart_eval(expression: str):
        """
        Best-effort evaluation for small expressions.
        - Tries SymPy first (safer than eval for many cases).
        - Falls back to a restricted eval for basic Python arithmetic.
        """
        expr = (expression or "").strip()
        if not expr:
            return None

        # SymPy: handles many math expressions, factorial, binomial, etc.
        try:
            allowed = {
                **sp.__dict__,
                **np.__dict__,
                **math.__dict__,
            }
            val = sp.sympify(expr, locals=allowed)
            # Try to fully evaluate to int if possible
            if val.is_integer():
                return int(val)
            if val.is_number:
                return float(val.evalf())
            return val
        except Exception:
            pass

        # Restricted eval fallback (no builtins).
        try:
            return eval(
                expr,
                {"__builtins__": {}},
                {**math.__dict__, **sp.__dict__, **np.__dict__},
            )
        except Exception:
            return None

    @staticmethod
    def generate_small_cases(expr_in_n: str, n_max: int = 8):
        """Evaluate an expression pattern by substituting n=1..n_max."""
        out = {}
        for n in range(1, max(1, n_max) + 1):
            try:
                expr = expr_in_n.replace("n", str(n))
                out[n] = AIMOToolkit.smart_eval(expr)
            except Exception:
                continue
        return out

    @staticmethod
    def pattern_analyzer(seq):
        """Detect simple patterns in a numeric sequence."""
        seq = list(seq)
        patterns = []
        if len(seq) < 2:
            return patterns

        # Arithmetic progression
        try:
            diff = seq[1] - seq[0]
            if all(seq[i + 1] - seq[i] == diff for i in range(len(seq) - 1)):
                patterns.append(("arithmetic", f"a_n = {seq[0]} + {diff}*(n-1)"))
        except Exception:
            pass

        # Geometric progression
        try:
            if seq[0] != 0:
                ratio = seq[1] / seq[0]
                if all(seq[i] != 0 and seq[i + 1] / seq[i] == ratio for i in range(len(seq) - 1)):
                    patterns.append(("geometric", f"a_n = {seq[0]} * ({ratio})^(n-1)"))
        except Exception:
            pass

        # Quadratic fit as a hint (works when seq is numeric)
        try:
            x = np.arange(1, len(seq) + 1)
            y = np.array(seq, dtype=float)
            coeffs = np.polyfit(x, y, 2)
            patterns.append((
                "quadratic_fit",
                f"a_n ≈ {coeffs[0]:.6g} n^2 + {coeffs[1]:.6g} n + {coeffs[2]:.6g}",
            ))
        except Exception:
            pass

        return patterns


def add_libs(code: str) -> str:
    """Add common math libraries and helpers to code."""
    return _TOOL_PRELUDE + code


def _is_probably_expr(line: str) -> bool:
    """Heuristic: is this line a standalone expression we can print?"""
    s = line.strip()
    if not s:
        return False
    if s.startswith(("#", "import ", "from ", "def ", "class ", "@")):
        return False
    if s.endswith(":"):
        return False
    if s.startswith(("return ", "pass", "break", "continue", "raise ")):
        return False
    try:
        ast.parse(s, mode="eval")
        return True
    except Exception:
        return False


def ensure_last_print(code: str) -> str:
    """Ensure the last standalone expression is printed (safe heuristic)."""
    lines = code.splitlines()
    # Find last non-empty, non-comment line
    idx = None
    for i in range(len(lines) - 1, -1, -1):
        if lines[i].strip() and not lines[i].lstrip().startswith("#"):
            idx = i
            break
    if idx is None:
        return code

    last = lines[idx]
    if "print(" in last:
        return code
    if _is_probably_expr(last):
        lines[idx] = f"print({last.strip()})"
        return "\n".join(lines)
    return code


class LocalJupyterSession:
    """Stateful Jupyter kernel session for code execution."""

    # Class-level lock and port counter to avoid port conflicts
    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:
        """Get next available ports for kernel connection."""
        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count
            return ports

    def __init__(self, connection_file: str | None = None, *, timeout: float = 120.0):
        try:
            from jupyter_client import BlockingKernelClient, KernelManager
        except ImportError as exc:
            raise RuntimeError("jupyter_client package required") from exc

        self._default_timeout = timeout
        self._owns_kernel = False
        self._client: BlockingKernelClient
        self._km: KernelManager | None = None

        if connection_file:
            from pathlib import Path
            connection_path = Path(connection_file).expanduser()
            if not connection_path.exists():
                raise FileNotFoundError(f"Connection file not found: {connection_path}")
            client = BlockingKernelClient()
            client.load_connection_file(str(connection_path))
            client.start_channels()
            client.wait_for_ready(timeout=self._default_timeout)
            self._client = client
        else:
            # Allocate unique ports to avoid conflicts when running multiple kernels
            ports = self._get_next_ports(5)
            km = KernelManager()
            km.shell_port = ports[0]
            km.iopub_port = ports[1]
            km.stdin_port = ports[2]
            km.hb_port = ports[3]
            km.control_port = ports[4]
            km.start_kernel()
            client = km.blocking_client()
            client.start_channels()
            client.wait_for_ready(timeout=self._default_timeout)
            self._client = client
            self._km = km
            self._owns_kernel = True

    def execute(self, code: str, *, timeout: float | None = None) -> str:
        """Execute code and return combined stdout/stderr."""
        client = self._client
        effective_timeout = timeout or self._default_timeout
        msg_id = client.execute(code, store_history=True, allow_stdin=False, stop_on_error=False)

        stdout_parts: list[str] = []
        stderr_parts: list[str] = []

        while True:
            try:
                msg = client.get_iopub_msg(timeout=effective_timeout)
            except queue.Empty as exc:
                raise TimeoutError("Timed out waiting for kernel output.") from exc

            if msg.get("parent_header", {}).get("msg_id") != msg_id:
                continue

            msg_type = msg.get("msg_type")
            content = msg.get("content", {})

            if msg_type == "stream":
                text = content.get("text", "")
                if content.get("name") == "stdout":
                    stdout_parts.append(text)
                else:
                    stderr_parts.append(text)
            elif msg_type == "error":
                traceback_data = content.get("traceback")
                if traceback_data:
                    stderr_parts.append("\n".join(traceback_data))
                else:
                    ename = content.get("ename", "")
                    evalue = content.get("evalue", "")
                    stderr_parts.append(f"{ename}: {evalue}".strip())
            elif msg_type in {"execute_result", "display_data"}:
                data = content.get("data", {})
                text = data.get("text/plain")
                if text:
                    stdout_parts.append(text if text.endswith("\n") else f"{text}\n")
            elif msg_type == "status" and content.get("execution_state") == "idle":
                break

        # Drain shell channel
        while True:
            try:
                reply = client.get_shell_msg(timeout=effective_timeout)
            except queue.Empty as exc:
                raise TimeoutError("Timed out waiting for execution reply.") from exc

            if reply.get("parent_header", {}).get("msg_id") != msg_id:
                continue

            reply_content = reply.get("content", {})
            if reply_content.get("status") == "error":
                traceback_data = reply_content.get("traceback")
                if traceback_data:
                    stderr_parts.append("\n".join(traceback_data))
                else:
                    ename = reply_content.get("ename", "")
                    evalue = reply_content.get("evalue", "")
                    stderr_parts.append(f"{ename}: {evalue}".strip())
            break

        stdout = "".join(stdout_parts)
        stderr = "".join(stderr_parts)

        if stderr:
            stdout = f"{stdout.rstrip()}\n{stderr}" if stdout else stderr

        if not stdout.strip():
            stdout = "[WARN] No output. Use print() to see results."

        return stdout

    def close(self):
        import contextlib
        with contextlib.suppress(Exception):
            self._client.stop_channels()
        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)

    def __del__(self):
        self.close()


class PythonTool:
    """Python execution tool using Jupyter kernel."""

    def __init__(self, execution_backend: str | None = None, local_jupyter_timeout: float = 60.0):
        self._local_jupyter_timeout = local_jupyter_timeout
        self._execution_lock = threading.Lock()
        self._jupyter_session: LocalJupyterSession | None = None
        # Lazy initialization to avoid port conflicts during object creation
        self._init_lock = threading.Lock()

    def _ensure_session(self):
        """Lazily initialize the Jupyter session."""
        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = LocalJupyterSession(timeout=self._local_jupyter_timeout)

    @classmethod
    def get_tool_name(cls) -> str:
        return "python"

    @property
    def name(self) -> str:
        return self.get_tool_name()

    @property
    def instruction(self) -> str:
        return """Use this tool to execute Python code. The code runs in a stateful Jupyter notebook. Use print() to see output."""

    @property
    def tool_config(self) -> ToolNamespaceConfig:
        return ToolNamespaceConfig(
            name=self.get_tool_name(),
            description=self.instruction,
            tools=[],
        )

    def _make_response(self, output: str, channel: str | None = None) -> Message:
        content = TextContent(text=output)
        author = Author(role=Role.TOOL, name=self.get_tool_name())
        message = Message(author=author, content=[content]).with_recipient("assistant")
        if channel:
            message = message.with_channel(channel)
        return message

    def process_sync_plus(self, message: Message) -> list[Message]:
        """Execute code from message using Jupyter kernel."""
        self._ensure_session()
        script = message.content[0].text
        script = ensure_last_print(add_libs(script))
        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(script)
            except TimeoutError as exc:
                output = f"[ERROR] {exc}"
        return [self._make_response(output, channel=message.channel)]

    def close(self):
        if self._jupyter_session is not None:
            self._jupyter_session.close()
            self._jupyter_session = None

    def __del__(self):
        self.close()

# Imports and Setup

In [ ]:
import warnings
warnings.simplefilter('ignore')

import re
import math
import threading
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List

import pandas as pd
import polars as pl
from openai import OpenAI
from transformers import set_seed, AutoTokenizer

from openai_harmony import (
    HarmonyEncodingName,
    load_harmony_encoding,
    Conversation,
    Message,
    Role,
    SystemContent,
    ReasoningEffort,
    RenderConversationConfig,
 )

from local_python_tool import PythonTool

# Load Harmony encoding for GPT-OSS
encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)

# Constants
SEED = 42
set_seed(SEED)
MAX_LEN = 64 * 1024
USE_BUDGET = False
K = 8  # Number of parallel samples

# Inference parameters (same as way-to-30 reference)
TEMPERATURE = 1.0
TOP_P = 1.0
MIN_P = 0.02

# Globals (initialized for robustness / static analysis)
cutoff_times = []
ground_truth = {}
predictions = {}
correct_count = 0
total_count = 0

# Start vLLM Server

In [ ]:
def start_vllm_server() -> subprocess.Popen:
    """Start vLLM server in background."""
    command = [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", "/kaggle/input/gpt-oss-120b/transformers/default/1",
        "--served-model-name", "gpt-oss",
        "--tensor-parallel-size", "1",
        "--max-num-seqs", "64",
        "--gpu-memory-utilization", "0.96",
        "--host", "0.0.0.0",
        "--port", "8000",
        "--dtype", "auto",
        "--max-model-len", str(MAX_LEN),
        "--stream-interval", "20",
    ]
    with open("./vllm.log", "w") as logfile:
        process = subprocess.Popen(
            command, stdout=logfile, stderr=subprocess.STDOUT, start_new_session=True
        )
    print("vLLM server started. Logs: ./vllm.log")
    return process


vllm_process = start_vllm_server()

# TIR Prompts

In [ ]:
import random

# Base instruction: keep submission-safe (answer only)
TIR_FINAL_INSTRUCTION = (
    "Finally, return only the verified final answer in \\boxed{}, "
    "where the answer is an integer in [0, 99999]. Never guess."
 )

# Prompt templates (compact but structured)
MATH_PROMPT_TEMPLATES = {
    "structured_cot": (
        "Problem: {problem}\n\n"
        "Solve step-by-step:\n"
        "1. UNDERSTANDING\n2. OBSERVATION\n3. STRATEGY\n4. SMALL_CASES (use python tool)\n"
        "5. PATTERN\n6. CONJECTURE\n7. PROOF\n8. VERIFICATION (use python tool)\n9. ANSWER\n\n"
        "Important: When you want to run code, call the python tool (do not put code in markdown fences).\n"
        + TIR_FINAL_INSTRUCTION
    ),
    "socratic_dialogue": (
        "Problem: {problem}\n\n"
        "Imagine a brief dialogue (student/professor) to guide discovery.\n"
        "You MUST call the python tool for computations / small cases.\n\n"
        + TIR_FINAL_INSTRUCTION
    ),
    "proof_by_contradiction": (
        "Problem: {problem}\n\n"
        "Try proof by contradiction when applicable. Use python tool for checks.\n\n"
        + TIR_FINAL_INSTRUCTION
    ),
    "inductive_reasoning": (
        "Problem: {problem}\n\n"
        "Try induction when a parameter n / recurrence / sequence appears. Use python tool.\n\n"
        + TIR_FINAL_INSTRUCTION
    ),
    "generating_function": (
        "Problem: {problem}\n\n"
        "Try generating functions / counting when it is a counting problem. Use python tool for algebra checks.\n\n"
        + TIR_FINAL_INSTRUCTION
    ),
}


def select_prompt_strategy(problem_text: str) -> str:
    """Heuristic prompt strategy selection."""
    s = (problem_text or "").lower()
    if any(w in s for w in ["prove", "show", "demonstrate"]):
        return "proof_by_contradiction"
    if any(w in s for w in ["sequence", "series", "recurrence"]):
        return "inductive_reasoning"
    if any(w in s for w in ["count", "number of ways", "arrangements", "permutation", "combination"]):
        return "generating_function"
    if any(w in s for w in ["for all", "every", "always"]):
        return "structured_cot"
    # default: structured (works broadly)
    return "structured_cot"


def build_user_prompt(problem_text: str, strategy: str) -> str:
    template = MATH_PROMPT_TEMPLATES.get(strategy, MATH_PROMPT_TEMPLATES["structured_cot"])
    # Tag the strategy so we can parse it back from raw transcripts if needed
    return template.format(problem=problem_text) + f"\n\n[STRATEGY:{strategy}]"

# Inferencer with Harmony Protocol

In [ ]:
class HarmonyTIRInferencer:
    """Inferencer using Harmony protocol with TIR (Tool-Integrated Reasoning)."""

    def __init__(
        self,
        model_path: str,
        max_model_len: int = MAX_LEN,
        temperature: float = TEMPERATURE,
        top_p: float = TOP_P,
        min_p: float = MIN_P,
        seed: int = SEED,
        k: int = K,
        use_budget: bool = USE_BUDGET,
        max_iter: int = 100,
    ):
        self.model_path = model_path
        self.model = "gpt-oss"
        self.max_model_len = max_model_len
        self.temperature = temperature
        self.top_p = top_p
        self.min_p = min_p
        self.seed = seed
        self.k = k
        self.use_budget = use_budget
        self.max_iter = max_iter
        self.base_budget = 60 * 5.5  # 5.5 minutes
        self.budget = 370
        self.deadline = None

        self.client = OpenAI(
            base_url="http://127.0.0.1:8000/v1",
            api_key="sk-local",
            timeout=360,
        )
        self.stop_token_ids = encoding.stop_tokens_for_assistant_actions()
        self.tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

    def wait_server(self):
        """Wait for vLLM server to be ready."""
        for _ in range(15 * 60):
            time.sleep(1)
            try:
                print(self.client.models.list())
                return
            except Exception:
                continue
        raise RuntimeError("vLLM server failed to start")

    def get_num_samples(self) -> int:
        """Get number of samples based on budget."""
        if not self.use_budget:
            print(f"Budget disabled -> N: {self.k}")
            return self.k
        estimated = (self.budget - 190) / 90
        ret = min(self.k, math.floor(estimated))
        print(f"Budget: {self.budget} -> N: {ret}")
        return max(1, ret)

    def apply_chat_template(self, prompt: str, python_tool: PythonTool) -> list[Message]:
        """Create Harmony messages with system prompt and tools."""
        return [
            Message.from_role_and_content(
                Role.SYSTEM,
                SystemContent.new()
                .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
                .with_tools(python_tool.tool_config),
            ),
            Message.from_role_and_content(Role.USER, prompt),
        ]

    def format_prompts(self, problem: str) -> list[str]:
        """Format problem with adaptive prompt strategies."""
        num_samples = self.get_num_samples()
        base_strategy = select_prompt_strategy(problem)
        # Keep some diversity while remaining deterministic per run
        rng = random.Random(self.seed + (hash(problem) & 0xFFFF))
        candidate_strategies = [
            base_strategy,
            "structured_cot",
            "socratic_dialogue",
            "inductive_reasoning",
            "proof_by_contradiction",
            "generating_function",
        ]
        # De-duplicate while preserving order
        seen = set()
        candidate_strategies = [s for s in candidate_strategies if not (s in seen or seen.add(s))]

        prompts = []
        for i in range(num_samples):
            # Choose a strategy (biased toward base_strategy)
            if i == 0:
                strategy = base_strategy
            else:
                # sample among first few candidates for diversity
                strategy = rng.choice(candidate_strategies[: min(3, len(candidate_strategies))])
            prompts.append(build_user_prompt(problem, strategy))
        return prompts

    def _sample_seeds(self, n: int, problem: str) -> list[int]:
        """Deterministic per-problem seeds for sample diversity."""
        base = self.seed + (hash(problem) & 0x7FFFFFFF)
        return [base + 9973 * i for i in range(n)]

    def inference(self, problem: str, deadline: float) -> int:
        """Run inference on a problem."""
        self.deadline = deadline
        start_time = time.time()

        prompts = self.format_prompts(problem)
        seeds = self._sample_seeds(len(prompts), problem)
        responses = self._inference_parallel(prompts, seeds)

        duration = time.time() - start_time
        print(f"[inference] Took {duration:.2f}s")

        if self.use_budget:
            budget_left = max(0, self.budget - duration)
            self.budget = self.base_budget + budget_left
            print(f"[inference] Updated budget: {self.budget:.2f}s")

        return self.parse_responses(responses)

    def single_generate_tir(self, prompt: str, stop_event: threading.Event, seed: int) -> str:
        """Generate single TIR response with tool execution."""
        python_tool = None
        try:
            python_tool = PythonTool(execution_backend="jupyter")
            messages = self.apply_chat_template(prompt, python_tool)
            final_answer_found = ""

            for iteration in range(self.max_iter):
                # Check termination conditions
                if self.deadline and time.time() >= self.deadline:
                    print("⏰ Deadline reached")
                    break
                if final_answer_found:
                    break
                if stop_event and stop_event.is_set():
                    print("🛑 Stop signal received")
                    break

                # Render conversation to token IDs
                prompt_ids = encoding.render_conversation_for_completion(
                    Conversation.from_messages(messages), Role.ASSISTANT
                )
                max_tokens = self.max_model_len - len(prompt_ids)
                if max_tokens < 1:
                    print("⚠️ Context full")
                    break

                token_buffer = []
                token_buffer_str = ""
                breaking = False

                # Stream generation
                stream = self.client.completions.create(
                    model=self.model,
                    prompt=prompt_ids,
                    max_tokens=max_tokens,
                    temperature=self.temperature,
                    top_p=self.top_p,
                    seed=seed,
                    stream=True,
                    extra_body=dict(
                        min_p=self.min_p,
                        stop_token_ids=self.stop_token_ids,
                        return_token_ids=True,
                    ),
                    timeout=360,
                )

                for chunk in stream:
                    if stop_event and stop_event.is_set():
                        breaking = True
                        break

                    token_chunk = chunk.choices[0].token_ids
                    text_chunk = chunk.choices[0].text

                    if token_chunk:
                        token_buffer.extend(token_chunk)
                        token_buffer_str += text_chunk

                    if self.deadline and time.time() >= self.deadline:
                        breaking = True
                        break

                    if len(token_buffer) > 60_000:
                        print("⚠️ Token limit")
                        breaking = True
                        break

                    # Check for boxed answer
                    if "}" in text_chunk and self.extract_boxed_text(token_buffer_str) is not None:
                        final_answer_found = token_buffer_str
                        breaking = True
                        break

                stream.close()

                if breaking:
                    break

                # Parse generated tokens into messages
                if token_buffer:
                    new_messages = encoding.parse_messages_from_completion_tokens(
                        token_buffer, Role.ASSISTANT
                    )
                    messages.extend(new_messages)

                    last_message = messages[-1]

                    # Check if generation is complete
                    if last_message.channel == "final" or token_buffer[-1] == 200002:
                        break

                    # Check if model wants to call python tool
                    if last_message.recipient == "python":
                        print("🐍 Executing Python code...")
                        response_msgs = python_tool.process_sync_plus(last_message)
                        messages.extend(response_msgs)

            # Return final response
            if final_answer_found:
                return final_answer_found

            # Render full conversation
            return encoding.decode_utf8(
                encoding.render_conversation_for_training(
                    Conversation.from_messages(messages),
                    RenderConversationConfig(auto_drop_analysis=False)
                )
            )

        except Exception as e:
            print(f"Error in generation: {e}")
            return ""
        finally:
            if python_tool:
                python_tool.close()

    def _inference_parallel(self, prompts: list[str], seeds: list[int]) -> list[str]:
        """Run parallel inference with early stopping."""
        stop_event = threading.Event()
        answers_collected = []
        raw_responses = [""] * len(prompts)
        majority_threshold = len(prompts) / 2

        print(f"🚀 Sampling {len(prompts)} times (threshold: > {majority_threshold})...")

        executor = ThreadPoolExecutor(max_workers=self.k)
        try:
            future_to_idx = {
                executor.submit(self.single_generate_tir, p, stop_event, seeds[i]): i
                for i, p in enumerate(prompts)
            }

            for future in as_completed(future_to_idx):
                idx = future_to_idx[future]
                try:
                    result_text = future.result()
                    raw_responses[idx] = result_text

                    ans = self.extract_boxed_text(result_text)
                    if ans is not None:
                        answers_collected.append(ans)
                        counts = Counter(answers_collected)
                        most_common_ans, count = counts.most_common(1)[0]

                        if count > majority_threshold:
                            print(f"🎯 Majority reached! {most_common_ans} appeared {count} times")
                            stop_event.set()
                            break
                except Exception as e:
                    print(f"Task exception: {e}")
        finally:
            executor.shutdown(wait=False, cancel_futures=True)

        return raw_responses

    def extract_boxed_text(self, text: str) -> int | None:
        """Extract answer from \\boxed{} or 'final answer is' patterns."""
        # Try \boxed{} pattern
        pattern = r"\\boxed\{(.*?)\}"
        matches = re.findall(pattern, str(text), flags=re.DOTALL)
        if matches:
            for match in reversed(matches):
                if match:
                    try:
                        clean_match = match.strip().replace(",", "").replace(" ", "")
                        # Accept simple integers (possibly as float-like '123.0')
                        val = int(float(clean_match[:32]))
                        if 0 <= val <= 99999:
                            return val
                    except Exception:
                        pass

        # Back-compat: original 'oxed{...}' pattern (in case of missing backslash)
        pattern = r"oxed\{(.*?)\}"
        matches = re.findall(pattern, str(text), flags=re.DOTALL)
        if matches:
            for match in reversed(matches):
                if match:
                    try:
                        clean_match = match.strip().replace(",", "").replace(" ", "")
                        val = int(float(clean_match[:32]))
                        if 0 <= val <= 99999:
                            return val
                    except Exception:
                        pass

        # Try 'final answer is X' pattern
        pattern = r"(?i)final\s+answer\s*(?:is|:)?\s*(\d{1,10})"
        matches = re.findall(pattern, str(text))
        if matches:
            for match in reversed(matches):
                if match:
                    try:
                        val = int(match)
                        if 0 <= val <= 99999:
                            return val
                    except Exception:
                        pass

        return None

    def _extract_strategy_tag(self, text: str) -> str | None:
        m = re.search(r"\[STRATEGY:([a-z_]+)\]", str(text))
        return m.group(1) if m else None

    def _extract_confidence(self, solution_text: str) -> float:
        """Heuristic confidence score from solution transcript."""
        s = (solution_text or "").lower()
        score = 0.5

        # Positive signals
        if "python" in s and ("execut" in s or "tool" in s):
            score += 0.1
        if "verify" in s or "verification" in s or "checked" in s:
            score += 0.1
        if "therefore" in s or "qed" in s:
            score += 0.05
        if "contradiction" in s:
            score += 0.05
        if "pattern" in s and "small" in s:
            score += 0.05

        # Negative signals
        if "guess" in s or "not sure" in s or "maybe" in s:
            score -= 0.15
        if "error" in s and "traceback" in s:
            score -= 0.2

        # Clip
        if score < 0.05:
            score = 0.05
        if score > 0.95:
            score = 0.95
        return score

    def parse_responses(self, responses: list[str]) -> int:
        """Parse responses and return a confidence-weighted consensus answer."""
        pairs = []  # (answer, weight, strategy)
        for r in responses:
            ans = self.extract_boxed_text(r)
            if ans is None:
                continue
            conf = self._extract_confidence(r)
            strat = self._extract_strategy_tag(r) or "unknown"
            # Small bonus for diversity strategies agreeing
            pairs.append((ans, conf, strat))

        if not pairs:
            print("No valid answers found")
            return 0

        # Weighted vote
        votes: dict[int, float] = {}
        strat_support: dict[int, set[str]] = {}
        for ans, w, strat in pairs:
            votes[ans] = votes.get(ans, 0.0) + float(w)
            strat_support.setdefault(ans, set()).add(strat)

        # Cross-method validation bonus: answers supported by >=2 strategies get +0.05
        for ans, strategies in strat_support.items():
            if len(strategies) >= 2:
                votes[ans] += 0.05

        # Report
        print("Weighted votes:", {k: round(v, 3) for k, v in sorted(votes.items(), key=lambda x: -x[1])[:5]})

        best = max(votes.items(), key=lambda x: x[1])[0]
        return int(best) % 100000

In [ ]:
inferencer = HarmonyTIRInferencer(
    "/kaggle/input/gpt-oss-120b/transformers/default/1",
    use_budget=USE_BUDGET,
    k=K,
)

In [ ]:
inferencer.wait_server()

# Submission

In [ ]:
# Cutoff schedule is computed after loading the reference file (depends on number of questions).
cutoff_times = []

In [ ]:
def predict(id_: pl.DataFrame, question: pl.DataFrame) -> pl.DataFrame | pd.DataFrame:
    """Make a prediction."""
    global correct_count, total_count, predictions, cutoff_times
    
    question_id = id_.item(0)
    question_text = question.item(0)

    print("------")
    print(f"ID: {question_id}")
    print(f"Question: {question_text[:200]}...")

    # Robust deadline handling: if schedule is missing/empty, fall back to global cutoff.
    if cutoff_times:
        current_deadline = cutoff_times[-1]
        cutoff_times.pop()
    else:
        current_deadline = final_cutoff_time

    answer = inferencer.inference(question_text, deadline=current_deadline)

    # Store prediction
    predictions[question_id] = answer
    
    # Check accuracy if ground truth available
    total_count += 1
    if question_id in ground_truth:
        gt = ground_truth[question_id]
        is_correct = (answer == gt)
        if is_correct:
            correct_count += 1
        status = "✅" if is_correct else "❌"
        print(f"Answer: {answer} | Ground Truth: {gt} | {status}")
        print(f"📊 Running Accuracy: {correct_count}/{total_count} ({100*correct_count/total_count:.1f}%)")
    else:
        print(f"Answer: {answer}")
    
    print("------\n")

    return pl.DataFrame({"id": question_id, "answer": answer})

In [ ]:
# Load reference data and keep ground truth for accuracy calculation
df = pd.read_csv(
    "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv"
 )

# Store ground truth answers for accuracy calculation (only in local mode)
ground_truth = dict(zip(df["id"], df["answer"])) if "answer" in df.columns else {}

# Create input file without answers
df.drop("answer", axis=1, errors="ignore").to_csv("reference.csv", index=False)

# Track predictions for accuracy calculation
predictions = {}
correct_count = 0
total_count = 0

# Compute a cutoff schedule based on the number of questions (robust vs hardcoded 50).
init_time = time.time()
n_questions = int(len(df)) if df is not None else 50
cutoff_times = [int(x) for x in np.linspace(final_cutoff_time, init_time, n_questions + 1)]
cutoff_times.pop()

In [ ]:
import kaggle_evaluation.aimo_3_inference_server

inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    inference_server.serve()
else:
    inference_server.run_local_gateway(("reference.csv",))
    
    # Print final accuracy summary
    if ground_truth and total_count > 0:
        print("\n" + "=" * 50)
        print("📊 FINAL ACCURACY SUMMARY")
        print("=" * 50)
        print(f"Correct: {correct_count}/{total_count}")
        print(f"Accuracy: {100*correct_count/total_count:.1f}%")
        print("=" * 50)
        
        # Show details
        print("\nDetails:")
        for qid, pred in predictions.items():
            if qid in ground_truth:
                gt = ground_truth[qid]
                status = "✅" if pred == gt else "❌"
                print(f"  {qid}: pred={pred}, gt={gt} {status}")